# 07 · Vectores, similitud y cierre

## Propósito

La última capa. Hasta acá spaCy nos dijo qué es cada palabra —categoría, lema,
función sintáctica—. Los vectores dicen algo distinto: **con qué otras palabras
se parece**.

Es también el puente hacia lo que sigue en la materia: los modelos de lenguaje
que vamos a ver después están construidos sobre esta misma idea, mucho más
grande.

Y es el cuaderno donde vamos a intentar algo, **va a salir mal**, y vamos a
entender por qué.

## Qué consume

- `datos/corpus.jsonl`
- `datos/corpus.spacy`
- `componentes.py`

## Celda de reanudación

In [ ]:
# spacy: la librería de análisis lingüístico.
import spacy

# pandas: para trabajar el corpus como una tabla.
import pandas as pd

# DocBin: el contenedor de documentos procesados.
from spacy.tokens import DocBin

# Nuestros componentes propios.
import componentes

# pipeline_propio: la herramienta que configuraste y guardaste en el cuaderno 05
nlp = spacy.load("pipeline_propio")
# lines=True: el archivo .jsonl tiene un registro JSON por renglón
corpus = pd.read_json("datos/corpus.jsonl", lines=True)

# Creamos un banco vacío y lo llenamos con lo que guardó el cuaderno 06.
banco = DocBin()
banco.from_disk("datos/corpus.spacy")

# get_docs() pide el vocab del pipeline para volver a armar cada Doc.
generador = banco.get_docs(nlp.vocab)
documentos = list(generador)

cantidad_documentos = len(documentos)
# :, separa los miles con una coma
print("documentos recuperados:", f"{cantidad_documentos:,}")

cantidad_parrafos = len(corpus)
if cantidad_documentos != cantidad_parrafos:
    print("ATENCIÓN: los documentos guardados no coinciden con los párrafos del corpus.")
    print("Volvé a correr el 06: es el que produce datos/corpus.spacy.")

## 1. Qué es un vector de palabra

Cada palabra del vocabulario tiene asociada una lista de números. Esa lista se
aprendió mirando en qué contextos aparece la palabra, y tiene una propiedad
útil: **palabras que se usan en contextos parecidos quedan cerca**.

In [ ]:
doc_perro = nlp("perro")
token = doc_perro[0]  # [0]: el primer (y único) token

# vector: la lista de números del token. shape: su tamaño
print("dimensiones:", token.vector.shape)

# [:8]: los primeros ocho números. round(3): cada uno con tres decimales
primeros_ocho = token.vector[:8]
primeros_ocho_redondeados = primeros_ocho.round(3)
print("primeros 8 números:", primeros_ocho_redondeados)

# vector_norm: el largo del vector. float(): lo pasa a un número común de Python
norma = float(token.vector_norm)
norma_redondeada = round(norma, 2)  # round(..., 2): con dos decimales
print("norma:", norma_redondeada)

300 números por palabra. No significan nada por separado; lo que significa
es la **distancia entre dos de esas listas**.

### Los tres modelos, otra vez

En el cuaderno `04` viste que `sm` tiene 0 vectores, `md` tiene 20.000 y `lg`
tiene 500.000. Ahora se ve por qué importa.

In [ ]:
# es_core_news_sm: el modelo chico, para comparar
chico = spacy.load("es_core_news_sm")

# vocab.vectors: la tabla de vectores del modelo. shape[0]: cuántas filas (palabras) tiene
tabla_sm = chico.vocab.vectors
tabla_lg = nlp.vocab.vectors
vectores_sm = tabla_sm.shape[0]
vectores_lg = tabla_lg.shape[0]

print("vectores en es_core_news_sm:", vectores_sm)
print("vectores en es_core_news_lg:", vectores_lg)
print()

# La misma cuenta de la norma que hicimos arriba, ahora con el modelo chico.
doc_perro_sm = chico("perro")
token_perro_sm = doc_perro_sm[0]
norma_sm = float(token_perro_sm.vector_norm)
norma_sm_redondeada = round(norma_sm, 2)
print("norma del vector de 'perro' en sm:", norma_sm_redondeada)

Cuidado con esa última línea, porque es una trampa: **`sm` tiene cero vectores
y sin embargo devuelve un vector con norma distinta de cero.**

Lo que devuelve no es un vector de palabra: es el tensor de contexto que produce
el componente `tok2vec`. La diferencia se ve así.

In [ ]:
def vector_de_banco(modelo, frase):
    """Devuelve el vector del token "banco" en esa frase."""
    documento_de_la_frase = modelo(frase)
    for token_actual in documento_de_la_frase:
        if token_actual.text == "banco":
            # return: devuelve el vector y termina la función, sin mirar el resto
            return token_actual.vector
    # Si llegamos acá, la frase no tenía "banco".
    return None


frase_plaza = "El banco de la plaza estaba roto."
frase_plata = "Fui al banco a sacar plata."

modelos = {"sm": chico, "lg": nlp}

for etiqueta in ["sm", "lg"]:
    modelo = modelos[etiqueta]
    vector_plaza = vector_de_banco(modelo, frase_plaza)
    vector_plata = vector_de_banco(modelo, frase_plata)
    # list(): pasa cada vector a una lista común. ==: ¿son iguales número por número?
    iguales = list(vector_plaza) == list(vector_plata)
    print(f"  {etiqueta}: mismo vector para 'banco' en las dos frases -> {iguales}")

En `lg`, `banco` tiene **un** vector, siempre el mismo: es una entrada fija de
una tabla de 500.000 palabras.

En `sm` cambia según la oración, porque no sale de una tabla sino de una red que
mira el contexto. Suena mejor —¡distingue el banco de plaza del banco de plata!—
pero para lo que vamos a hacer acá es inservible: no se pueden comparar dos
palabras si cada aparición tiene su propio vector.

spaCy avisa, con un aviso que es facilísimo pasar por alto.

In [ ]:
doc_perro_sm = chico("perro")
doc_gato_sm = chico("gato")

# Si spaCy tiene algo que avisar, el aviso sale abajo, en un recuadro aparte.
# Ojo: Python lo muestra una sola vez. Si corrés la celda de nuevo, ya no aparece.
resultado = doc_perro_sm.similarity(doc_gato_sm)
resultado_redondeado = round(resultado, 3)
print("similitud perro-gato con sm:", resultado_redondeado)

Devolvió un número perfectamente creíble, calculado sobre algo que no son
vectores de palabra. **Cualquier similitud calculada con `sm` es ruido con buena
presentación.**

## 2. Similitud

`similarity()` compara dos vectores y devuelve un número entre 0 y 1.

In [ ]:
# Cada pareja es un diccionario con las dos palabras a comparar.
parejas = [
    {"palabra_a": "perro", "palabra_b": "gato"},
    {"palabra_a": "madre", "palabra_b": "padre"},
    {"palabra_a": "perro", "palabra_b": "computadora"},
    {"palabra_a": "fútbol", "palabra_b": "cocina"},
]

filas = []
for pareja in parejas:
    palabra_a = pareja["palabra_a"]
    palabra_b = pareja["palabra_b"]
    doc_a = nlp(palabra_a)
    doc_b = nlp(palabra_b)
    # similarity(): qué tan parecidos son los vectores de los dos documentos
    similitud = doc_a.similarity(doc_b)
    fila = {
        "palabra_a": palabra_a,
        "palabra_b": palabra_b,
        "similitud": round(similitud, 3),
    }
    filas.append(fila)

pd.DataFrame(filas)

Funciona como uno esperaría: *perro* y *gato* cerca, *fútbol* y *cocina*
lejos.

Pero mirá esto.

In [ ]:
# es_core_news_md: el modelo mediano
mediano = spacy.load("es_core_news_md")

# vocab["manzana"]: la entrada de esa palabra en el vocabulario del modelo
vector_manzana_md = mediano.vocab["manzana"].vector
vector_pera_md = mediano.vocab["pera"].vector
vector_manzana_lg = nlp.vocab["manzana"].vector
vector_pera_lg = nlp.vocab["pera"].vector

# list(): pasa cada vector a una lista común. ==: ¿son iguales número por número?
mismo_vector_md = list(vector_manzana_md) == list(vector_pera_md)
mismo_vector_lg = list(vector_manzana_lg) == list(vector_pera_lg)

print("¿'manzana' y 'pera' tienen el MISMO vector?")
print()
print("  en md:", mismo_vector_md)
print("  en lg:", mismo_vector_lg)
print()

busqueda_md = mediano("fruta")
busqueda_lg = nlp("fruta")

palabras_de_prueba = ["manzana", "pera", "plátano", "naranja", "mesa", "teclado"]

filas = []
for palabra in palabras_de_prueba:
    doc_md = mediano(palabra)
    doc_lg = nlp(palabra)
    similitud_md = busqueda_md.similarity(doc_md)
    similitud_lg = busqueda_lg.similarity(doc_lg)
    fila = {
        "palabra": palabra,
        "md": round(similitud_md, 3),
        "lg": round(similitud_lg, 3),
    }
    filas.append(fila)

pd.DataFrame(filas)

**En `md`, *manzana* y *pera* comparten literalmente la misma fila de la tabla
de vectores.** Por eso dan idéntico al tercer decimal.

No es un bug: el modelo mediano tiene 20.000 vectores únicos para cientos de
miles de palabras, así que agrupa. El grande tiene 500.000 y las separa.

Guardate el caso. Es el mejor ejemplo de por qué un número que parece razonable
puede estar producido por algo que no tiene nada que ver con tu pregunta.

### Palabras fuera del vocabulario

`token.is_oov` dice si la palabra está en la tabla de vectores. Probemos con
léxico rioplatense.

In [ ]:
palabras = ["boludo", "laburo", "pibe", "quilombo", "bondi", "morfar", "asado", "ñandubay"]

filas = []
for palabra in palabras:
    doc_prueba = nlp(palabra)
    token_prueba = doc_prueba[0]
    norma_del_token = float(token_prueba.vector_norm)
    fila = {
        "palabra": palabra,
        # is_oov: True si la palabra no está en la tabla de vectores (oov: out of vocabulary)
        "is_oov": token_prueba.is_oov,
        "norma_del_vector": round(norma_del_token, 1),
    }
    filas.append(fila)

pd.DataFrame(filas)

Con `lg` casi todo el léxico rioplatense está cubierto. Con `md` no sería
así, y con `sm` no habría nada.

Es otra razón concreta para justificar los 631 MB.

## 3. Un intento que sale mal

Ahora la pregunta interesante: **¿los años del corpus se parecen entre sí?**
¿Escribía distinto Casciari en 2004 y en 2015?

El vector de un `Doc` es el promedio de los vectores de sus palabras. Podemos
juntar todos los párrafos de un año en un solo `Doc` y comparar los años entre sí
con `similarity()`, igual que comparamos palabras. Parece razonable.

In [ ]:
# Doc: la clase de los documentos de spaCy
from spacy.tokens import Doc

# Agrupamos los párrafos por año.
parrafos_por_anio = {}

anios = list(corpus["anio"])

# posicion: en qué párrafo vamos. En esa misma posición de anios está su año.
posicion = 0
for documento in documentos:
    anio = int(anios[posicion])  # int(): lo pasamos a número entero común
    if anio not in parrafos_por_anio:
        parrafos_por_anio[anio] = []
    parrafos_por_anio[anio].append(documento)
    posicion = posicion + 1

# Doc.from_docs(): pega todos los párrafos de un año en un solo Doc.
# exclude=["user_data"]: deja afuera nuestras extensiones, que para esto no hacen falta
documento_por_anio = {}
for anio in parrafos_por_anio:
    parrafos_del_anio = parrafos_por_anio[anio]
    documento_por_anio[anio] = Doc.from_docs(parrafos_del_anio, exclude=["user_data"])

# keys(): los años. sorted(): ordenados de menor a mayor
anios_de_los_documentos = documento_por_anio.keys()
anios_con_documento = sorted(anios_de_los_documentos)
print("años con documento:", anios_con_documento)

In [ ]:
similitudes_entre_anios = []
for anio_a in anios_con_documento:
    fila = []
    for anio_b in anios_con_documento:
        documento_a = documento_por_anio[anio_a]
        documento_b = documento_por_anio[anio_b]
        # similarity(): la misma que usaste con palabras, ahora entre dos años enteros
        valor = documento_a.similarity(documento_b)
        valor_redondeado = round(valor, 3)
        fila.append(valor_redondeado)
    similitudes_entre_anios.append(fila)

# index= y columns=: los años, como nombres de las filas y de las columnas
pd.DataFrame(similitudes_entre_anios, index=anios_con_documento, columns=anios_con_documento)

## 4. Por qué salió mal

Mirá los números: **todos están entre 0,97 y 1,00.** El año más parecido a 2004
y el más distinto están separados por dos centésimas.

Eso no es un resultado. Es un instrumento que no discrimina.

La razón es aritmética y vale la pena entenderla: **promediar muchísimos vectores
converge al promedio del corpus.** Cada año tiene entre 116 y 813 párrafos, o sea
miles de palabras, y al promediarlas todas, las particularidades se cancelan entre
sí y lo que queda es "español escrito por Casciari" en los doce casos.

Comprobemos que el instrumento sí funciona cuando no se promedia tanto.

In [ ]:
# random: para elegir párrafos de forma reproducible.
import random

# Nos quedamos con párrafos de largo parecido, para que la comparación sea justa.
candidatos = []
# posicion: en qué párrafo vamos. Guardamos la posición, no el documento.
posicion = 0
for documento in documentos:
    largo = len(documento.text)
    # Entre 300 y 600 caracteres: más de 300...
    if largo > 300:
        # ...y menos de 600.
        if largo < 600:
            candidatos.append(posicion)
    posicion = posicion + 1

# seed(7): fija el azar. Con la misma semilla salen siempre los mismos párrafos
random.seed(7)
# sample(): elige al azar, sin repetir. k=5: cinco párrafos
elegidos = random.sample(candidatos, k=5)

print("párrafos elegidos:", elegidos)

Cinco párrafos dan diez pares. Los comparamos todos contra todos, sin repetir: por
eso el segundo índice arranca después del primero.

In [ ]:
cantidad_elegidos = len(elegidos)

filas = []
# primero y segundo: lugares dentro de la lista elegidos (0 es el primero).
# range(a, b): los números desde a hasta b, sin incluir b.
for primero in range(cantidad_elegidos):
    for segundo in range(primero + 1, cantidad_elegidos):
        # elegidos guarda posiciones del corpus: con ellas buscamos cada documento
        posicion_a = elegidos[primero]
        posicion_b = elegidos[segundo]
        documento_a = documentos[posicion_a]
        documento_b = documentos[posicion_b]
        similitud = documento_a.similarity(documento_b)
        fila = {
            "parrafo_a": posicion_a,
            "parrafo_b": posicion_b,
            "similitud": round(similitud, 3),
        }
        filas.append(fila)

pd.DataFrame(filas)

Entre 0,63 y 0,87: acá sí hay variación para leer.

**La lección no es "los vectores no sirven".** Es que el nivel de agregación al
que hacés la cuenta decide si vas a ver algo o no. Con 5.428 párrafos promediados
de a 400, no ves nada. Con párrafos sueltos, sí.

Y también: si hubiéramos armado un mapa de calor lindo con la matriz de años,
habría parecido un resultado. Los colores no avisan que el rango es de dos
centésimas.

> ### Perilla
>
> `similarity()` **siempre** devuelve un número entre 0 y 1. Nunca devuelve
> "no sé", nunca devuelve "esta comparación no tiene sentido".
>
> Que dos textos den 0,87 no significa nada por sí solo. Significa algo
> **comparado con** otras parejas del mismo corpus, calculadas con el mismo
> modelo y al mismo nivel de agregación.
>
> Antes de reportar cualquier similitud, preguntate: ¿cuál es el rango de valores
> que da este instrumento sobre estos datos? Si es de dos centésimas, no tenés
> un resultado.

## 5. El otro problema de los vectores

Recién viste un problema **aritmético**: promediar muchos vectores converge al promedio
del corpus y borra las diferencias. Es cierto, y es la mitad de la historia.

La otra mitad no tiene que ver con la cuenta sino con **de dónde salieron los números**.

Los vectores de `es_core_news_lg` se entrenaron con fastText sobre OSCAR Common Crawl y
Wikipedia. O sea: sobre texto que ya estaba escrito. Un vector no es una definición de la
palabra, es un resumen de **con qué otras palabras suele aparecer**.

Entonces, si en esos textos ciertas palabras aparecen juntas más seguido que otras, esa
regularidad queda adentro del vector. Y todo lo que calcules después la hereda.

In [ ]:
def parecido(palabra_a, palabra_b):
    """Similitud entre dos palabras sueltas, según los vectores del modelo.

    Devuelve None si alguna de las dos no tiene vector.
    """
    # nlp.vocab[palabra]: la entrada de la palabra en el vocabulario (un "lexema").
    # Para una palabra suelta no hace falta procesar un texto entero.
    lexema_a = nlp.vocab[palabra_a]
    lexema_b = nlp.vocab[palabra_b]

    # has_vector: True si la palabra tiene vector. Si alguna no tiene, no hay qué comparar.
    if not lexema_a.has_vector:
        return None
    if not lexema_b.has_vector:
        return None

    similitud = lexema_a.similarity(lexema_b)
    return round(similitud, 3)


parecido_enfermera = parecido("enfermera", "cuidar")
parecido_enfermero = parecido("enfermero", "cuidar")
print("enfermera / cuidar:", parecido_enfermera)
print("enfermero / cuidar:", parecido_enfermero)

Mismo procedimiento que la prueba del `04`: pares mínimos. Cada oficio en femenino y en
masculino, y la distancia de cada uno a dos verbos.

Lo único que cambia entre las dos columnas de un par es el género gramatical.

In [ ]:
OFICIOS = [
    {"femenino": "enfermera", "masculino": "enfermero"},
    {"femenino": "maestra", "masculino": "maestro"},
    {"femenino": "ingeniera", "masculino": "ingeniero"},
    {"femenino": "secretaria", "masculino": "secretario"},
    {"femenino": "jefa", "masculino": "jefe"},
]

cantidad_de_pares = len(OFICIOS)
print("pares en la prueba:", cantidad_de_pares)

Y ahora los cinco pares juntos.

In [ ]:
verbos_de_referencia = ["cuidar", "dirigir"]

filas = []
for oficio in OFICIOS:
    femenino = oficio["femenino"]
    masculino = oficio["masculino"]
    fila = {"par": femenino + " / " + masculino}
    for referencia in verbos_de_referencia:
        # Armamos los nombres de las dos columnas, por ejemplo "ella + cuidar".
        columna_ella = "ella + " + referencia
        columna_el = "él + " + referencia
        fila[columna_ella] = parecido(femenino, referencia)
        fila[columna_el] = parecido(masculino, referencia)
    filas.append(fila)

pd.DataFrame(filas)

### Cómo se lee, y la trampa que ya conocés

Otra vez: **no te digo el resultado.** Corré la celda y mirála antes de seguir leyendo.

Y acá se aplica exactamente la regla que sacaste hace dos secciones. Si la diferencia
entre *«ella + cuidar»* y *«él + cuidar»* es de dos centésimas, **no tenés un resultado**.
Es el mismo instrumento con el mismo problema de rango.

Fijate tres cosas antes de concluir nada:

| Pregunta | Por qué importa |
|---|---|
| ¿Cuál es el rango total de la tabla? | Si todos los valores caen entre 0,30 y 0,34, no hay señal |
| ¿La diferencia va siempre para el mismo lado? | Una diferencia chica pero **consistente** en los cinco pares dice más que una grande y aislada |
| ¿Algún par se comporta al revés? | Los contraejemplos son información, no molestias |

### Lo que dio cuando la corrimos

| par | ella + cuidar | él + cuidar | ella + dirigir | él + dirigir |
|---|---|---|---|---|
| enfermera / enfermero | 0,301 | **0,351** | 0,160 | **0,189** |
| maestra / maestro | 0,156 | **0,164** | 0,259 | **0,307** |
| ingeniera / ingeniero | **0,154** | 0,053 | **0,326** | 0,281 |
| secretaria / secretario | **0,068** | 0,013 | 0,318 | **0,340** |
| jefa / jefe | 0,008 | **0,022** | 0,191 | **0,336** |

En negrita, el más alto de cada par.

Ahora aplicá el criterio: **¿la diferencia va siempre para el mismo lado?**

No. Con *cuidar*, el masculino queda más cerca en tres pares de cinco y el femenino en los
otros dos. Con *dirigir*, el masculino en cuatro de cinco, pero *ingeniera* rompe la
serie. Y las diferencias dentro de un par van de 0,008 a 0,10, con todo el rango de la
tabla entre 0,008 y 0,351.

**No hay patrón consistente, así que no hay resultado.** La prueba no encontró lo que
fuimos a buscar, y decirlo es parte del trabajo.

Es la segunda vez en la unidad que pasa: en el `04` la prueba de apellidos tampoco
confirmó lo que sospechábamos. Que no confirme **no la vuelve inútil**. Una prueba que
solo puede darte la razón no es una prueba: es una ilustración.

> Y ojo con la tentación de insistir. Si probás quince combinaciones de palabras hasta que
> una da la diferencia que buscabas, esa diferencia no significa nada: la encontraste
> porque buscáste mucho, no porque esté ahí.

### Lo que esto significa, y lo que no

Los vectores no tienen opiniones. Nadie los programó para asociar nada con nadie.

Lo que hacen es reproducir **coocurrencias**: qué palabras aparecen cerca de qué otras en
los textos con que se entrenaron. Si un oficio quedara sistemáticamente más cerca de un
verbo según su género, eso no diría nada sobre las personas que ejercen ese oficio: diría
qué se escribió, y cuánto, en Common Crawl y en Wikipedia.

**El problema no es que el vector opine. Es que no opina y parece un dato objetivo.**

Un número con tres decimales que salió de una matriz tiene aspecto de medición neutral.
No lo es: es un promedio de lo que la gente escribió en internet, con todo lo que eso
arrastra. Y a diferencia de un error de NER, que se ve, esto viaja escondido dentro de
cada similitud que calcules.

> ### Perilla
>
> Probá con otros verbos y otros pares. *«Limpiar»*, *«mandar»*, *«cocinar»*, *«programar»*.
> Pares como *«doctora / doctor»* o *«presidenta / presidente»*.
>
> **Escribí qué esperás antes de correr cada uno.**
>
> Y después la pregunta difícil: si encontrás una diferencia, ¿qué hacés con ella? No
> se puede arreglar el vector. Lo único que se puede hacer es **declararlo** cuando
> reportás cualquier resultado que dependa de esos vectores.

## 6. Ejercicio

1. Buscá los tres párrafos más parecidos a uno que elijas vos. Leelos. ¿Se
   parecen de verdad, o el número te engañó?
2. Rehacé la comparación entre años pero **sin promediar**: tomá 20 párrafos
   sueltos de 2004 y 20 de 2015, y compará todos contra todos. ¿Las parejas del
   mismo año dan más alto que las de años distintos?
3. Buscá diez palabras del corpus que sean `is_oov` con `md` pero no con `lg`.

In [ ]:
# TU CÓDIGO ACÁ
#
# Punto 2: comparar párrafos sueltos de 2004 y de 2015, sin promediar.
#
#   Paso 1: juntá en una lista las posiciones de los párrafos de 2004. Recorré
#           documentos con el contador posicion, como en la celda de los
#           candidatos. El año de cada párrafo está en anios[posicion].
#           Hacé lo mismo con 2015.
#
#   Paso 2: fijá el azar con random.seed(7) y elegí 20 de cada año con
#               random.sample(posiciones_2004, k=20)
#
#   Paso 3: juntá las 40 posiciones en una sola lista y compará todos contra todos
#           con los dos for de la celda de las parejas. Sumale a cada fila el
#           año de cada párrafo.
#
#   Paso 4: separá las filas en "mismo año" y "años distintos", y compará el
#           promedio de la similitud de cada grupo.

## 7. Antes de irte: cómo se formula un título-tesis

En el integrador vas a tener que partir de una afirmación, no de un tema. La diferencia
tiene nombre y conviene practicarla ahora, con las operaciones que acabás de aprender.

### Título-etiqueta contra título-tesis

| | |
|---|---|
| **Título-etiqueta** | *Análisis morfológico de los textos de Casciari* |
| **Título-tesis** | *Entre 2004 y 2015, los textos de Casciari duplican la proporción de verbos en pretérito imperfecto, y el salto se concentra a partir de 2010* |

El primero anuncia sobre qué vas a mirar. **No afirma nada, así que no puede estar
equivocado**, y por eso tampoco puede estar bien.

El segundo dice algo que puede ser falso. Esa es toda la diferencia.

### Qué tiene que tener

Un título-tesis se reconoce porque se le pueden señalar cuatro cosas:

| | En el ejemplo |
|---|---|
| **Una variable que se mide** | proporción de verbos en pretérito imperfecto |
| **Una comparación** | 2004 contra 2015 |
| **Una dirección y una magnitud** | duplican |
| **Algo que lo refutaría** | que la proporción no cambie, o que baje |

Si no podés completar la cuarta fila, todavía no tenés una tesis: tenés un tema.

### La pregunta que hay que hacerse

> **¿Qué resultado me haría decir que estaba equivocado?**

Escribila antes de correr nada. Si no hay ningún resultado posible que te haga cambiar de
idea, lo que tenés no se puede poner a prueba, y entonces el análisis va a confirmar lo
que ya pensabas hicieras lo que hicieras.

### Con lo que sabés de esta unidad

Cada cuaderno habilita un tipo de afirmación distinto:

| Operación | Un título-tesis posible |
|---|---|
| Lema y POS (`02`) | *El vocabulario de X se concentra en menos lemas a medida que avanza la serie* |
| Morfología (`02`) | *Los textos de Y usan primera persona el doble que los de Z* |
| Sintaxis (`03`) | *Las oraciones de la sección política son más largas y más subordinadas que las de deportes* |
| Entidades (`03`, `05`) | *En la cobertura de W, los actores institucionales aparecen tres veces más que las personas* |

> ### Perilla
>
> Escribí **un** título-tesis sobre un material que te interese, con las cuatro filas
> completas.
>
> Después hacé la prueba difícil: decí en voz alta qué tendrías que encontrar para
> tener que abandonarlo. Si no se te ocurre nada, reescribilo.

## Cierre de la unidad

Recorriste el camino completo.

**Del texto a los datos.** Un corpus que no era texto, que perdió sus límites de
obra y su procedencia, reconstruido hasta donde se podía y documentado hasta
donde había que documentarlo.

**Las capas.** Tokens, oraciones, lemas, categorías gramaticales, morfología,
sintaxis, entidades, vectores. Cada una contesta preguntas que las otras no
pueden.

**La herramienta configurada.** Reglas propias, componentes propios, atributos
propios, guardados en disco y reutilizables.

**La escala.** Un corpus completo procesado una vez y guardado, para no volver a
esperar.

Pero si hay una sola cosa que quiero que te lleves, es esta:

> **En cada cuaderno de esta unidad, el modelo se equivocó.**
>
> `EE.UU.` pegado a la oración siguiente. Los adjetivos coordinados que no
> aparecían. *Editorial Orsai* como lugar. `¿` como lugar, treinta y tres veces.
> *tenés* con lema *tenés*. *Mirta*, *Bertotti* y *Mirta Bertotti* contados como
> tres personas. *manzana* y *pera* con el mismo vector. Doce años que se parecen
> todos entre sí al 99%.
>
> **Ninguno de esos errores hizo saltar una excepción.** Todos devolvieron un
> resultado con buena cara. La única forma de encontrarlos fue mirar la salida y
> compararla con lo que sabíamos del texto.

spaCy es un instrumento, y como todo instrumento tiene un rango donde funciona y
supuestos incorporados. El modelo aprendió con periodismo peninsular y vos lo
usaste sobre un blog rioplatense. Eso no lo invalida: lo vuelve algo que hay que
calibrar y sobre lo que hay que informar.

Las decisiones que tomaste —qué modelo, qué lemas corregir, qué nombres van en
la lista, qué componentes apagar, a qué nivel agregar— **no son configuración.
Son tu método**, y tienen que estar escritas y poder defenderse.

## Lo que viene

El `08` es el trabajo práctico: elegís un material, lo explorás y documentás el
recorrido. Ahí las perillas las movés vos.